#### Notebook: Write data to the Mongodb collections

In [ ]:
import platform
import datetime
import os

# Set proxy information if windows
if platform.system() == "Windows":
    print("Platform is Windows, setting proxy")
    # Get the current date and time
    now = datetime.datetime.now()
    day = now.strftime("%A")
    proxy_url = f"http://33566:{day[0:3]}@proxy-west.aero.org:8080"

    # Set proxy environment variables
    os.environ['HTTP_PROXY'] = proxy_url
    os.environ['HTTPS_PROXY'] = proxy_url

In [ ]:
from urllib.parse import quote_plus
import json
import sys
# Get the current working directory (notebooks directory)
current_dir = os.getcwd()

# Go up one level to the project directory
project_dir = os.path.dirname(current_dir)

# Assuming your project structure is as described before
src_path = os.path.abspath(os.path.join(project_dir, 'src'))

# Add the 'src' directory to the Python path
sys.path.append(src_path)

from question_answer_site.question_answer.parse_document import parse_document, update_collection
from question_answer_site.question_answer.mongodb import MongoDb
from question_answer_site.question_answer.config import TOKENIZER, EMBEDDING_MODEL_FNAME, EMBEDDING_MODEL_TYPE, TOKENS_EMBEDDINGS, DOCUMENT_EMBEDDING, \
    DOCUMENT_TOKENS, TOP_N, TRANSFORMER_MODEL_NAME, METHOD, MAX_QUERY_LENGTH, username, password, cluster_url, INPUT_FOLDER, \
    database_name, special_characters, CHUNK_SIZE, CHUNK_OVERLAP
from transformers import BertTokenizer, BertForQuestionAnswering, RobertaTokenizer, RobertaForQuestionAnswering

from gensim.models import Word2Vec
import spacy
import copy

In [ ]:
# Set the Tokenizer for your specific BERT model variant
bert_base_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
roberta_tokenizer = RobertaTokenizer.from_pretrained("deepset/roberta-base-squad2", add_prefix_space = True)

tokenizers = {'bert': bert_base_tokenizer, 'roberta': roberta_tokenizer}

tokenizer = tokenizers[TOKENIZER]

In [ ]:
# Load your trained Word2Vec model
if EMBEDDING_MODEL_TYPE == 'Word2Vec':
    embedding_model = Word2Vec.load(
        os.path.join(os.getcwd(), "question_answer", 
                     "embedding_models", EMBEDDING_MODEL_FNAME))
elif EMBEDDING_MODEL_TYPE.lower() == 'glove':
    # Load the custom spaCy model
    embedding_model = spacy.load(os.path.join("..","src","question_answer_site", "question_answer",
                                              "embedding_models", EMBEDDING_MODEL_FNAME.split(".bin")[0]))

In [ ]:
document_path = os.path.join("..", "data", "space_based_pdfs")

In [ ]:
data = parse_document(directory=document_path,
                      embedding_layer_model=embedding_model,
                      tokenizer=tokenizer,
                      chunk_size=CHUNK_SIZE,
                      chunk_overlap=CHUNK_OVERLAP,
                      additional_stopwords=special_characters)

In [ ]:
data[0].keys()

In [ ]:
update_collection("parsed_documents", copy.deepcopy(data))

In [ ]:
# Should be +1 for adding one document
update_collection("extracted_text", copy.deepcopy(data))